# tdep-view — quick tour

A short walkthrough of `tdep-view`: load a TDEP trajectory, analyse atomic
displacements, export to a standard format, and render a frame and a movie.

It runs against the small CsPbI₃ trajectory bundled in `../tests/data`.
Set the environment variable `TDEP_VIEW_PREFIX` to point at your own `infile`
prefix instead.

Rendering (the last two sections) needs the visualization extra:
`pip install ".[viz,movie]"`.

In [ ]:
import os
from pathlib import Path

# Locate the example trajectory (bundled data, or $TDEP_VIEW_PREFIX).
prefix = Path(os.environ.get('TDEP_VIEW_PREFIX',
                            Path.cwd().parent / 'tests' / 'data' / 'infile'))
prefix

## 1. Load and inspect

In [ ]:
from tdep.visualization import Trajectory

traj = Trajectory.from_prefix(prefix)
print('atoms      :', traj.natoms)
print('frames     :', traj.nframes)
print('timestep   :', traj.dt_fs, 'fs')
print('temperature:', traj.temperature, 'K')
print('has forces :', traj.forces is not None)
traj.symbols[:6]

## 2. Displacement analysis

Per-atom RMS vibration amplitude (reference-free), then the time-averaged
structure compared against the `ssposcar` reference to quantify static
distortion per species.

In [ ]:
import numpy as np
from tdep.visualization.analysis import vibration_rms

rms = vibration_rms(traj.scaled_positions, traj.cell.lattice)
print(f'RMS vibration: mean {rms.mean():.4f} Å, max {rms.max():.4f} Å')

cmp = traj.compare_average_to_reference()
print(f'averaged-vs-reference RMS deviation: {cmp.rms:.4f} Å')
cmp.per_species()

## 3. Export

Convert a strided subset to Extended XYZ (readable by OVITO and ASE).

In [ ]:
out_dir = Path('_output'); out_dir.mkdir(exist_ok=True)
stride = max(1, traj.nframes // 20)
traj.export(out_dir / 'traj.xyz', 'extxyz', stride=stride)
print('wrote', out_dir / 'traj.xyz')

## 4. Screenshot

Renders off-screen and shows the image inline. Needs the `[viz]` extra.

In [ ]:
from IPython.display import Image, display

try:
    uw = traj.unwrapped()
    shot = out_dir / 'frame0.png'
    uw.screenshot(shot, index=0, color_by='displacement', show_forces=True)
    display(Image(str(shot)))
except ImportError as exc:
    print('install the [viz] extra to render:', exc)

## 5. Inline movie

Render a short GIF and embed it. Needs `[viz]` (+ `[movie]` for the writer).

In [ ]:
try:
    mstride = max(1, traj.nframes // 60)
    gif = out_dir / 'traj.gif'
    uw.to_movie(gif, stride=mstride, fps=20, color_by='displacement')
    display(Image(str(gif)))
except ImportError as exc:
    print('install the [viz,movie] extras to render a movie:', exc)